In [ ]:
class PIController:
    #PI controller with selectable integration and anti-windup

    def __init__(
        self, Kp, Ki, dt,
        u_min=None, u_max=None,
        anti_windup=True,
        method="rect"
    ):
        if method not in ("rect", "trap"):
            raise ValueError("method must be 'rect' or 'trap'")

        self.Kp = Kp
        self.Ki = Ki
        self.dt = dt
        self.u_min = u_min
        self.u_max = u_max
        self.anti_windup = anti_windup
        self.method = method

        self.integral = 0.0
        self.prev_error = 0.0

    def _clamp(self, u):
        if self.u_max is not None:
            u = min(u, self.u_max)
        if self.u_min is not None:
            u = max(u, self.u_min)
        return u

    def update(self, setpoint, measurement):
        error = setpoint - measurement

        # Calculate the integral using the selected method
        if self.method == "rect":
            integral_change = self.Ki * error * self.dt
        else:  # trapezoidal integration
            integral_change = (
                self.Ki * (error + self.prev_error) * self.dt / 2
            )

        candidate = self.integral + integral_change

        # Calculate and limit the controller output
        u_unsat = self.Kp * error + candidate
        u = self._clamp(u_unsat)

        # Conditional integration anti-windup
        if self.anti_windup:
            winding_up = (
                (u_unsat > u and error > 0)
                or (u_unsat < u and error < 0)
            )

            if not winding_up:
                self.integral = candidate
        else:
            self.integral = candidate

        self.prev_error = error

        return u

    def reset(self):
        self.integral = 0.0
        self.prev_error = 0.0

Why store prev_error? Trapezoidal integration needs the error from the previous time step to calculate the average error. The reset() method clears it so the next run starts with a clean controller state.